# Zero-shot 난이도 probe (Qwen3.5-2B / 4B) — Colab

Colab에서 이 파일을 열어(파일 → 노트북 열기 → GitHub, 또는 Drive에 올려서) 런타임 유형을 GPU로 설정한 뒤 위에서부터 실행하세요. 데이터는 Drive의 `MyDrive/stv/data.zip`, 결과(npz·csv)는 `MyDrive/stv/outputs/`에 저장됩니다.

학습 없이, 모든 모델에 **같은 prompt · 같은 이미지 · 같은 a/b/c/d 보기**를 넣고 다음 토큰의 a/b/c/d logits로 `P(correct)`를 구해 train 샘플 난이도를 매깁니다.

| 결과 | 난이도 |
|---|---|
| 2B, 4B 모두 정답 | Easy |
| 2B 실패, 4B 정답 | Medium |
| 둘 다 실패 | Hard (모델 취약 유형) |
| 2B 정답, 4B 실패 | Inconsistent (운/노이즈 가능성) |

연속 점수: `difficulty = 1 - 가중평균 P(correct)` (기본은 동일 가중 = 평균 confidence).

**Sanity check**: dev는 사람도 2:3으로 갈린 모호한 문항이므로, 정상적인 probe라면 train 대비 dev에서 accuracy ↓, P(correct) ↓, prediction entropy ↑ 가 나와야 합니다.

> **모델 형식**: `unsloth/Qwen3.5-*-GGUF`는 llama.cpp 전용 포맷이라 transformers/unsloth에서 logits를 뽑을 수 없습니다. 같은 가중치의 safetensors 저장소 `unsloth/Qwen3.5-2B`, `unsloth/Qwen3.5-4B`를 **bf16(비양자화)** 로 로드합니다. 8B probe를 추가하려면 `PROBES`에 한 줄만 넣으면 됩니다(작은 모델 → 큰 모델 순서).

## 0. Colab 환경 준비
Drive 마운트 → repo clone → `scripts/setup.sh`(uv 환경 + 데이터 압축 해제). private repo라 clone 시 토큰이 필요합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
![ -d scene-text-vision-repo ] || git clone -b feat/baseline https://github.com/min9-less-min9-team/scene-text-vision-repo.git
%cd scene-text-vision-repo
!git pull
!bash scripts/setup.sh /content/drive/MyDrive/stv/data.zip

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path("/content/scene-text-vision-repo")
os.chdir(REPO); sys.path.insert(0, str(REPO / "eda"))
# 세션이 회수돼도 남도록 결과는 Drive에 저장
OUT_ROOT = Path("/content/drive/MyDrive/stv/outputs")

# 작은 모델 → 큰 모델 순서. 값 = (HF 모델 id, 결과 폴더, 배치 크기)
# 배치 크기는 16GB GPU(T4 포함) 기준의 안전한 값. prefill이 연산 병목이라 배치를 더 키워도 속도는 거의 같습니다.
# OOM이면 줄이고, 정확한 최대치는 `uv run stv-bench`로 확인하세요.
PROBES = {
    "2b": ("unsloth/Qwen3.5-2B", str(OUT_ROOT / "probe_2b"), 8),
    "4b": ("unsloth/Qwen3.5-4B", str(OUT_ROOT / "probe_4b"), 4),
}
SPLITS = ["train", "dev"]
LIMIT = 0        # 0 = 전체. 먼저 200 정도로 돌려 보고 0으로 바꾸세요 (LIMIT을 바꾸면 기존 npz를 지우고 다시 실행)

## 1. Zero-shot 추론
모델·split마다 별도 프로세스(`uv` 환경)로 실행해 끝나면 VRAM이 해제됩니다. 모델은 첫 실행 때 자동으로 받으며(2B ≈ 4.5GB, 4B ≈ 9GB) `scripts/env.sh`가 캐시를 Drive(`MyDrive/stv/hf_cache`)에 둡니다. 결과(`{split}_probs.npz`)가 이미 있으면 건너뛰므로 세션이 끊겨도 이어서 실행할 수 있습니다.
`--valid-size 0 --split train` = train.csv 전체, `--no-load-in-4bit` = bf16 비양자화, `--adapter-dir none` = zero-shot.

In [ ]:
RUN = ["bash", "-c", 'source scripts/env.sh; exec $STV_PY -m stv.inference "$@"', "_"]
for name, (model_id, out_dir, batch) in PROBES.items():
    for split in SPLITS:
        if (Path(out_dir) / f"{split}_probs.npz").exists():
            print(f"[skip] {name} {split}"); continue
        args = ["--model-id", model_id, "--adapter-dir", "none",
                "--no-load-in-4bit", "--split", split, "--valid-size", "0",
                "--max-infer-samples", str(LIMIT), "--infer-batch-size", str(batch), "--output-dir", out_dir]
        print("$ stv.inference", " ".join(args)); subprocess.run(RUN + args, cwd=REPO, check=True)

## 2. 샘플별 결과표

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from difficulty import difficulty_table, probe_summary, tier_counts, TIER_ORDER
from common import question_type

probe_dirs = {name: out_dir for name, (_, out_dir, _) in PROBES.items()}
# 가중 점수를 쓰려면 예: weights={"2b": 0.4, "4b": 0.6}
tables = {split: difficulty_table(probe_dirs, split) for split in SPLITS}
train, dev = tables["train"], tables["dev"]
train.head()

## 3. Sanity check: train vs dev
기대 패턴: dev에서 accuracy ↓, P(correct) ↓, entropy ↑. dev 정답은 annotator 다수결이며, 동률(is_tie) 문항을 뺀 값도 같이 봅니다.

In [ ]:
summary = pd.concat({
    "train": probe_summary(train, probe_dirs),
    "dev": probe_summary(dev, probe_dirs),
    "dev (동률 제외)": probe_summary(dev[~dev.is_tie], probe_dirs),
}).round(4)
display(summary)

for name in PROBES:
    t, d = summary.loc[("train", name)], summary.loc[("dev", name)]
    ok = t.accuracy > d.accuracy and t.p_correct_mean > d.p_correct_mean and t.entropy_mean < d.entropy_mean
    print(f"{name}: {'OK  - 기대 패턴과 일치' if ok else 'CHECK - 기대 패턴과 다름'}")

In [ ]:
fig, axes = plt.subplots(1, len(PROBES), figsize=(6 * len(PROBES), 3.5), sharey=True)
for ax, name in zip(np.atleast_1d(axes), PROBES):
    bins = np.linspace(0, 1, 21)
    ax.hist(train[f"p_correct_{name}"], bins=bins, density=True, alpha=0.6, label="train")
    ax.hist(dev[f"p_correct_{name}"], bins=bins, density=True, alpha=0.6, label="dev")
    ax.set_title(f"P(correct) - {name}"); ax.set_xlabel("P(correct)"); ax.legend()
plt.tight_layout(); plt.show()

## 4. 난이도 구분 (train)

In [ ]:
display(pd.concat({"train": tier_counts(train), "dev": tier_counts(dev)}, axis=1))

names = list(PROBES)
if len(names) == 2:
    a, b = names
    print("\n정답 여부 교차표 (train)")
    display(pd.crosstab(train[f"{a}_correct"], train[f"{b}_correct"], margins=True))

train.difficulty.hist(bins=40, figsize=(7, 3)); plt.title("train difficulty = 1 - mean P(correct)"); plt.show()

## 5. 어떤 문항이 어려운가

In [ ]:
train["question_type"] = train.question.map(question_type)
by_type = train.groupby("question_type").agg(
    n=("id", "size"), difficulty=("difficulty", "mean"),
    hard_pct=("tier", lambda s: (s == "Hard").mean() * 100),
    **{f"acc_{n}": (f"{n}_correct", "mean") for n in PROBES},
).sort_values("difficulty", ascending=False).round(3)
display(by_type)

# dev: 사람 합의율이 낮을수록 모델도 어려워하는가
print("dev: consensus_ratio vs difficulty 상관 =", round(dev.consensus_ratio.corr(dev.difficulty), 3))

cols = ["id", "question", "answer"] + [f"pred_{n}" for n in PROBES] + [f"p_correct_{n}" for n in PROBES] + ["difficulty"]
print("\n가장 어려운 train 문항 20개 (두 모델 모두 확신을 갖고 틀림 → 라벨 오류 후보이기도 함)")
display(train.sort_values("difficulty", ascending=False)[cols].head(20))

## 6. 저장
`MyDrive/stv/outputs/difficulty_{split}.csv` → 학습 시 난이도별 샘플링·가중치·오답 분석에 사용합니다. repo에 반영하려면 `eda/tables/`로 복사해 커밋하세요.

In [ ]:
for split, table in tables.items():
    path = OUT_ROOT / f"difficulty_{split}.csv"
    table.drop(columns=["question"]).to_csv(path, index=False)
    print("saved", path, len(table))